# The residual ℤ₂ of the aligned vacuum, and the seesaw neutrino sector

**A research notebook, not a tutorial.** It opens the LFV paper plan
([`ROADMAP_LFV.md`](ROADMAP_LFV.md)) at its spine, items B1, S1 and S5/R1. Notebooks 03,
07 and 08 found that μ→eγ and τ→eγ vanish at the exact-S₃ vacuum while τ→μγ does not,
and traced it to a "residual $\mathbb Z_2$" in prose. Here that symmetry is **derived**, its
consequence is stated as a selection rule for every channel, and the selection rule is
checked coupling by coupling against the stored scans.

Then the neutrinos. The residual $\mathbb Z_2$ cuts the electron out of the charged-lepton
mixing, so the electron mixing seen in oscillations must come from the neutrino sector. The
right-handed neutrinos the draft declares ([LFVHD]) are built here as a type-I seesaw, and
two questions are answered: does exact S₃ allow the observed mixing, and if the neutrino
sector must break the $\mathbb Z_2$, does that breaking leak back into charged-lepton LFV?

## What this notebook establishes

1. **The generator** (§1): exactly one non-identity element of S₃ fixes the aligned vacuum,
   a reflection. The potential and every Yukawa structure are invariant under it; the four
   soft quadratics split into one $\mathbb Z_2$-even and one $\mathbb Z_2$-odd combination per
   pair.
2. **The parities** (§2) of all physical states, read off the rotations: the electron and
   the in-plane non-vacuum scalars ($h_0$, one $A$, one $H^\pm$) are odd.
3. **The selection rule** (§3–§4): every coupling with odd total parity is zero at every
   exact point, and the channel table follows.
4. **The seesaw sector** (§5–§6): at exact S₃ the PMNS matrix has a single non-zero entry
   in the electron row, so **exact S₃ cannot produce solar or reactor mixing**, the lepton
   analogue of $V_{us}=0$ (notebook 02).
5. **The minimal fix** (§7–§8): a $\mathbb Z_2$-odd soft breaking of the Majorana mass alone
   reproduces the oscillation data exactly, leaving the charged-lepton and scalar sectors,
   and hence the selection rule, untouched at tree level.
6. **No leak** (§9): the induced charged-lepton LFV through heavy-neutrino loops is
   suppressed by $m_\nu/M_R$ for any seesaw scale; at least six orders below MEG II even at a
   tuned benchmark.

In [ ]:
import json, math, sys, time, itertools
from pathlib import Path

import numpy as np
import sympy as sp
from scipy.optimize import least_squares

sys.path.insert(0, str(Path.cwd()))
from feynlag import Vacuum, check_discrete_invariance
from model import build_model, rotation
import fermions as F
import lfv
import z2
import neutrinos as N
import radiative as RD
import soft_decays as S
import decays as D
import store
from sm_inputs import ALPHA, G_F
import derive as dv

sp.init_printing(use_latex="mathjax")
ok = dv.ok


def check_quoted(section, rows):
    "Assert that the numbers the markdown quotes still match this run."
    stale = [(what, got, want) for what, got, want, tol in rows if abs(float(got) - float(want)) > tol]
    assert not stale, "§%s prose is stale (what, computed, quoted): %s" % (section, stale)
    ok("the %d numbers quoted in §%s match this run" % (len(rows), section))


ledger = dv.Ledger("09 — the residual ℤ₂ and the seesaw neutrino sector")
t0 = time.time()
m = build_model()
H1, H2, HS = m.doublets
vac2 = [m.v1.s.subs(m.align), m.v2.s]
print("exact-S3 model built in %.1f s; aligned doublet vacuum (v1, v2) =" % (time.time() - t0), vac2)

exact-S3 model built in 0.5 s; aligned doublet vacuum (v1, v2) = [sqrt(3)*v_2/3, v_2]


---
## 1. The generator

The aligned vacuum is a direction in the S₃ doublet plane. Which group elements leave it
fixed? The six elements are built from feynlag's two generators and each is tested; nothing
about the answer is typed in.

In [ ]:
elements = z2.s3_doublet_elements(m.s3)
fixing = z2.vacuum_fixing_elements(m.s3, vac2)
print("elements fixing the vacuum:", fixing)
gname, g, G3 = z2.residual_generator(m.s3, vac2)
dv.show("g = %s =" % gname, g)
assert fixing == ["e", gname] and sp.simplify(g * g) == sp.eye(2) and sp.simplify(g.det()) == -1
ok("exactly one non-identity element fixes the vacuum: a reflection (det -1, g^2 = 1), so a Z2 survives")

elements fixing the vacuum: ['e', 'ab']
g = ab =   [1 terms]


⎡      √3 ⎤
⎢-1/2  ── ⎥
⎢      2  ⎥
⎢         ⎥
⎢ √3      ⎥
⎢ ──   1/2⎥
⎣ 2       ⎦

✓ exactly one non-identity element fixes the vacuum: a reflection (det -1, g^2 = 1), so a Z2 survives


**Recognise.** A reflection whose mirror line is the vacuum direction. The vacuum breaks S₃
down to this $\mathbb Z_2$, the "residual" symmetry; nothing is assumed about its action on
fields beyond the S₃ assignment of each multiplet.

**Invariance.** The potential is S₃-invariant, so it must be invariant under $g$ in
particular. Checked directly: $g$ acts on both doublet components,
$(H_1, H_2) \to g(H_1, H_2)$.

In [ ]:
def doublet_action(model):
    a, b = model.doublets[0], model.doublets[1]
    sub = {}
    for c in range(2):
        x1, x2 = a.components[c], b.components[c]
        sub[x1] = g[0, 0] * x1 + g[0, 1] * x2
        sub[x2] = g[1, 0] * x1 + g[1, 1] * x2
        sub[sp.conjugate(x1)] = sp.conjugate(sub[x1])
        sub[sp.conjugate(x2)] = sp.conjugate(sub[x2])
    return sub


V = m.model.lagrangian.total
dv.check_invariant(V, doublet_action(m), "the S3 potential is invariant under the residual reflection",
                   simplifier=sp.expand)

ms = build_model(soft=True, soft_solve_for="mD1sq")
Vsoft = sp.expand(ms.model.lagrangian.total)
soft_struct = {n: Vsoft.coeff(p.s) for n, p in ms.softs.items()}
T_D = z2.doublet_bilinear_rep(g)
dv.show("the soft pair (m_D1^2, m_D2^2) transforms with T =", T_D)
dirs = z2.even_odd_directions(T_D)
dv.show("Z2-even coefficient direction:", dirs[1].T)
dv.show("Z2-odd coefficient direction: ", dirs[-1].T)
# the S-doublet pair (H_S^dag H_i + h.c.) transforms with g itself
dirs_S = z2.even_odd_directions(g)
assert sp.simplify(dirs_S[1] - dirs[1]) == sp.zeros(2, 1)
ok("each soft pair has one Z2-even and one Z2-odd combination, along the same directions")

# the falsifier: a Z2-odd soft combination must FAIL the invariance check
odd_soft = dirs[-1][0] * soft_struct["mD1sq"] + dirs[-1][1] * soft_struct["mD2sq"]
even_soft = dirs[1][0] * soft_struct["mD1sq"] + dirs[1][1] * soft_struct["mD2sq"]
dv.check_invariant(even_soft, doublet_action(ms), "the Z2-even soft combination is invariant",
                   simplifier=sp.expand)
_ = dv.falsify("the Z2-odd soft combination", lambda: dv.check_invariant(
    odd_soft, doublet_action(ms), "odd soft term invariant?", simplifier=sp.expand))

✓ the S3 potential is invariant under the residual reflection


the soft pair (m_D1^2, m_D2^2) transforms with T =   [1 terms]


⎡      √3 ⎤
⎢-1/2  ── ⎥
⎢      2  ⎥
⎢         ⎥
⎢ √3      ⎥
⎢ ──   1/2⎥
⎣ 2       ⎦

Z2-even coefficient direction:   [1 terms]


⎡     √3⎤
⎢1/2  ──⎥
⎣     2 ⎦

Z2-odd coefficient direction:    [1 terms]


⎡-√3      ⎤
⎢────  1/2⎥
⎣ 2       ⎦

✓ each soft pair has one Z2-even and one Z2-odd combination, along the same directions
✓ the Z2-even soft combination is invariant
✗ not invariant (odd soft term invariant?); residual


         ___       ___          ___       ___       ___          ___       ___ ↪
- √3⋅H₁₀⋅H₁₀ - H₁₀⋅H₂₀ - √3⋅H₁ₚ⋅H₁ₚ - H₁ₚ⋅H₂ₚ - H₂₀⋅H₁₀ + √3⋅H₂₀⋅H₂₀ - H₂ₚ⋅H₁ₚ ↪

↪           ___
↪  + √3⋅H₂ₚ⋅H₂ₚ

✓ the Z2-odd soft combination — the check fires on a broken input, so it has teeth


**Recognise.** The soft terms are not all equal: in each pair one combination respects the
residual $\mathbb Z_2$ and one breaks it. Only the odd combinations (and the vacuum
misalignment they drive) can switch on the forbidden channels. This is the seed of the
order parameter the roadmap asks for (B2); here it is only recorded.

In [ ]:
_ = ledger.step(r"The residual $\mathbb Z_2$ of the aligned vacuum is the S₃ reflection $ab$ about the vacuum direction",
            obtained="derived",
            checks=("found by testing all six S₃ elements on the aligned vacuum: exactly one non-identity element fixes it",
                    "the potential is invariant under it (direct substitution)",
                    r"each soft pair splits into one $\mathbb Z_2$-even and one $\mathbb Z_2$-odd combination"),
            falsified_by=r"a $\mathbb Z_2$-odd soft combination passing the invariance check (falsify run)",
            section="1")

---
## 2. The parities of the physical states

A state has a definite parity if its flavour vector is an eigenvector of $G=\mathrm{diag}(g,1)$.
**Scalars:** the geometric rotation's columns (notebook 01) are tested, in all three sectors at
once since they share it. **Charged leptons:** the mass matrix is read off the Lagrangian
at the aligned vacuum, and $G^{\mathsf T}M_\ell G=M_\ell$ is checked; then its $G$-odd
eigenvector and eigenvalue are found.

In [ ]:
R = rotation(m)
col_parity = [z2.parity(G3, R[:, k]) for k in range(3)]
print("parities of the geometric rotation's columns (vacuum, in-plane, out-of-plane):", col_parity)
assert col_parity == [1, -1, 1]
ok("the vacuum direction and the out-of-plane direction are even; the in-plane non-vacuum direction is odd")

lep = F.build_lepton_sector(m.s3, m.SU2L, m.U1Y, m.doublets)
bars, rights = lep.mass_legs()
M_l = F.mass_matrix_from_bilinears(lep.lagrangian(), bars, rights, Vacuum(list(m.doublets)))
M_l = M_l.subs(m.align).applyfunc(sp.simplify)
assert (G3.T * M_l * G3 - M_l).applyfunc(sp.simplify) == sp.zeros(3, 3)
ok("G^T M_l G = M_l at the aligned vacuum: the charged-lepton mass matrix respects the Z2")
w_odd = R[:, 1]
m_odd = sp.simplify((w_odd.T * M_l * w_odd)[0])
assert (M_l * w_odd - m_odd * w_odd).applyfunc(sp.simplify) == sp.zeros(3, 1)
dv.show("the Z2-odd charged lepton has mass", m_odd)

parities of the geometric rotation's columns (vacuum, in-plane, out-of-plane): [1, -1, 1]
✓ the vacuum direction and the out-of-plane direction are even; the in-plane non-vacuum direction is odd


✓ G^T M_l G = M_l at the aligned vacuum: the charged-lepton mass matrix respects the Z2
the Z2-odd charged lepton has mass   [2 terms]


√2⋅Y_1__ell⋅v_S   √6⋅Y_2__ell⋅v₂
─────────────── + ──────────────
       2                3       

√2⋅Y_1__ell⋅v_S   √6⋅Y_2__ell⋅v₂
─────────────── + ──────────────
       2                3       

The odd scalars are the in-plane non-vacuum states: column 1 of $R$ in each sector, i.e.
notebook 03's gauge-phobic $h_0$, one $A$ and one $H^\pm$. Which lepton is the odd one is a
statement about the fit: the draft's lepton dictionary ([LFVHD], `lfv.aligned_yukawas`)
must put the electron there.

In [ ]:
Ts = S.build_tools()
exact_pts = store.points("viable_points")
SOFT0 = {n: 0.0 for n in ("mD1sq", "mD2sq", "mS1sq", "mS2sq")}
G3n = np.array(G3.evalf(), dtype=float)


def exact_rotations(pe):
    lp = {"lambdas": pe["lambdas"], "theta": pe["theta"], "phi": math.pi / 3, "softs_GeV2": SOFT0}
    return lp, S.point_rotations(Ts, lp, check=False)


lp0, rl0 = exact_rotations(exact_pts[0])
Yl0 = lfv.aligned_yukawas(Ts.g_of, 1.0, rl0["vevs"][0], rl0["vevs"][2])
masses0, O0, _ = lfv.signed_mass_basis(Ts.yuk_of(Yl0)["even"], rl0["vevs"])
lep_par = [z2.parity(G3n, O0[:, j]) for j in range(3)]
print("charged-lepton masses [GeV]:", masses0, " parities (e, mu, tau):", lep_par)
assert lep_par == [-1, 1, 1]
m_odd_num = float(m_odd.subs({**dict(zip(lep.coupling_symbols, Yl0)), m.v2.s: rl0["vevs"][1], m.vS.s: rl0["vevs"][2]}))
print("symbolic odd mass at this point: %.6e GeV  (m_e = %.6e)" % (abs(m_odd_num), masses0[0]))
assert abs(abs(m_odd_num) / masses0[0] - 1) < 1e-9
ok("the Z2-odd charged lepton IS the electron; mu and tau are even")

_ = ledger.step(r"Parities of the physical states: $e$, $h_0$, one $A$ and one $H^\pm$ are odd; $\mu$, $\tau$ and the other four scalars even",
            obtained="derived",
            checks=(r"rotation columns tested as $G$-eigenvectors: (vacuum, in-plane, out-of-plane) = (+, −, +)",
                    r"$G^{\mathsf T}M_\ell G=M_\ell$ symbolically; the odd eigenvalue equals $m_e$ at a fitted point to $10^{-9}$"),
            falsified_by="a mass eigenstate that is not a G-eigenvector (z2.parity raises)",
            section="2")

charged-lepton masses [GeV]: [5.10998950e-04 1.05658375e-01 1.77693000e+00]  parities (e, mu, tau): [-1, 1, 1]
symbolic odd mass at this point: 5.109990e-04 GeV  (m_e = 5.109990e-04)
✓ the Z2-odd charged lepton IS the electron; mu and tau are even


---
## 3. The selection rule

$G$ is a symmetry of the full Lagrangian, kinetic and gauge terms included (it is an element
of the flavour group, which commutes with the gauge group), and of the vacuum. So every
Green's function is $G$-invariant, and an amplitude whose external states carry parities
$P_i$ obeys

$$\mathcal A = \Big(\prod_i P_i\Big)\,\mathcal A \quad\Longrightarrow\quad \mathcal A = 0 \ \text{ if } \prod_i P_i = -1,$$

to all orders in perturbation theory. Photons, $Z$, $W$ and nuclei are even. The channel
table follows by counting:

In [ ]:
P = {"e": -1, "mu": 1, "tau": 1, "gamma": 1, "N": 1, "h_even": 1, "h_odd": -1}
channels = [("mu -> e gamma", ["mu", "e", "gamma"]), ("tau -> e gamma", ["tau", "e", "gamma"]),
            ("tau -> mu gamma", ["tau", "mu", "gamma"]), ("mu -> 3e", ["mu", "e", "e", "e"]),
            ("mu N -> e N", ["mu", "N", "e", "N"]), ("tau -> 3mu", ["tau", "mu", "mu", "mu"]),
            ("tau -> mu e e", ["tau", "mu", "e", "e"]), ("tau -> e mu mu", ["tau", "e", "mu", "mu"]),
            ("h_even -> e mu", ["h_even", "e", "mu"]), ("h_even -> e tau", ["h_even", "e", "tau"]),
            ("h_even -> tau mu", ["h_even", "tau", "mu"]), ("h_odd -> e mu", ["h_odd", "e", "mu"]),
            ("h_odd -> e tau", ["h_odd", "e", "tau"]), ("h_odd -> tau mu", ["h_odd", "tau", "mu"])]
table = {name: math.prod(P[x] for x in legs) for name, legs in channels}
print("%-18s %s" % ("channel", "at the exact vacuum"))
for name, par in table.items():
    print("%-18s %s" % (name, "allowed" if par > 0 else "FORBIDDEN (odd)"))
forbidden = sorted(k for k, v in table.items() if v < 0)
ok("%d of %d channels are forbidden to all orders" % (len(forbidden), len(table)))

channel            at the exact vacuum
mu -> e gamma      FORBIDDEN (odd)
tau -> e gamma     FORBIDDEN (odd)
tau -> mu gamma    allowed
mu -> 3e           FORBIDDEN (odd)
mu N -> e N        FORBIDDEN (odd)
tau -> 3mu         allowed
tau -> mu e e      allowed
tau -> e mu mu     FORBIDDEN (odd)
h_even -> e mu     FORBIDDEN (odd)
h_even -> e tau    FORBIDDEN (odd)
h_even -> tau mu   allowed
h_odd -> e mu      allowed
h_odd -> e tau     allowed
h_odd -> tau mu    FORBIDDEN (odd)
✓ 8 of 14 channels are forbidden to all orders


---
## 4. Checked coupling by coupling

The rule predicts which entries of every physical scalar's $3\times3$ lepton coupling matrix
vanish: those with $P_\phi P_a P_b=-1$. At the exact vacuum, for every 10th viable point of
notebook 01 and three values of $\mu_3$, every physical scalar's couplings in the lepton mass
basis are computed (notebook 07's machinery), each state's parity is read off **its own
eigenvector**, and the predicted zeros are checked. The allowed off-diagonal entries are
checked to be non-zero, so the test is not passed by couplings that vanish anyway.

In [ ]:
t0 = time.time()
worst_forbidden, smallest_allowed, n_eval = 0.0, np.inf, 0
for pe in exact_pts[::10]:
    lp, rl = exact_rotations(pe)
    par = {}
    for i, k in enumerate(("h1", "h2", "h3")):
        par[k] = z2.parity(G3n, rl["R_even"][:, i])
    for i, k in ((1, "A1"), (2, "A2")):
        par[k] = z2.parity(G3n, rl["R_odd"][:, i])
    for i, k in ((1, "Hpm1"), (2, "Hpm2")):
        par[k] = z2.parity(G3n, rl["R_C"][:, i])
    assert sorted(par.values()) == [-1, -1, -1, 1, 1, 1, 1]
    for mu3 in (0.3, 1.0, 1.7):
        Yl = lfv.aligned_yukawas(Ts.g_of, mu3, rl["vevs"][0], rl["vevs"][2])
        _, O, _ = lfv.signed_mass_basis(Ts.yuk_of(Yl)["even"], rl["vevs"])
        pl = [z2.parity(G3n, O[:, j]) for j in range(3)]
        _, Yp = S.mass_basis_yukawas(Ts, Yl, rl)
        for k, pk in par.items():
            for a, b in itertools.product(range(3), range(3)):
                val = abs(Yp[k][a, b])
                if pk * pl[a] * pl[b] < 0:
                    worst_forbidden = max(worst_forbidden, val)
                elif a != b:
                    smallest_allowed = min(smallest_allowed, val)
        n_eval += 1
print("%d (point, mu_3) evaluations in %.0f s" % (n_eval, time.time() - t0))
print("largest odd-parity coupling   : %.1e" % worst_forbidden)
print("smallest allowed off-diagonal : %.1e" % smallest_allowed)
assert worst_forbidden < 1e-14 and smallest_allowed > 1e-7
ok("every odd-parity coupling vanishes; every allowed off-diagonal coupling is non-zero")

ex = store.summary("tau_l_gamma")["summary"]["exact"]
print("notebook 08, all %d exact evaluations: max B(tau -> e gamma) = %.1e (round-off)"
      % (ex["n_evaluations"], ex["max_br_tau_e"]))
assert ex["max_br_tau_e"] < 1e-24

_ = ledger.step(r"Selection rule: an amplitude whose external states multiply to $-1$ vanishes to all orders",
            obtained="derived",
            statement=r"\prod_i P_i=-1\ \Rightarrow\ \mathcal A=0:\quad \mu\to e\gamma,\ \tau\to e\gamma,\ \mu\to3e,\ \mu N\to eN,\ h_{\rm even}\to e\ell,\ h_{\rm odd}\to\tau\mu \text{ forbidden}",
            checks=("every odd-parity scalar-lepton coupling < 1e-14 over 594 exact (point, mu3) evaluations, parities read per eigenvector",
                    "allowed off-diagonal couplings all non-zero (the test is not vacuous)",
                    "notebook 08's B(tau -> e gamma) is round-off at all 118 380 exact evaluations"),
            falsified_by="a non-zero coupling with odd total parity at any exact point",
            section="3-4")

594 (point, mu_3) evaluations in 0 s
largest odd-parity coupling   : 2.4e-15
smallest allowed off-diagonal : 3.3e-06
✓ every odd-parity coupling vanishes; every allowed off-diagonal coupling is non-zero
notebook 08, all 118380 exact evaluations: max B(tau -> e gamma) = 6.9e-34 (round-off)


---
## 5. The seesaw neutrino sector

Three right-handed neutrinos, $(\nu_{1R},\nu_{2R})$ a doublet and $\nu_{SR}$ a singlet, as
declared in [LFVHD]. They allow the five Dirac structures on $\tilde H$ (notebook 02 §8) and,
being gauge singlets, a Majorana mass $-\tfrac12\nu_R^{\mathsf T}CM_R\nu_R$. The invariant
$M_R$ is solved from the generators.

In [ ]:
nuR, nus = N.build_neutrino_sector(m.s3, m.U1Y, lep, m.doublets)
for k, term in nus.terms.items():
    assert check_discrete_invariance(term, m.s3)[0]
ok("all five Dirac structures on H-tilde pass feynlag's S3 check")

MR_inv = N.invariant_majorana_basis(m.s3)
dv.show("S3-invariant M_R =", MR_inv)
L_maj = N.majorana_lagrangian(nuR, MR_inv)
assert check_discrete_invariance(L_maj, m.s3)[0]
assert (N.majorana_matrix(L_maj, nuR) - MR_inv).applyfunc(sp.simplify) == sp.zeros(3, 3)
delta = sp.Symbol("delta")
broken = sp.zeros(3, 3); broken[0, 1] = broken[1, 0] = delta
dv.falsify("an S3-breaking Majorana entry", lambda: check_discrete_invariance(
    N.majorana_lagrangian(nuR, broken), m.s3)[0])
ok("M_R = diag(M, M, M_S): two invariant structures, read back off the MajoranaBilinear Lagrangian")

bars_n, rights_n = nus.mass_legs()
m_D = F.mass_matrix_from_bilinears(nus.lagrangian(), bars_n, rights_n, Vacuum(list(m.doublets)))
m_D = m_D.subs(m.align).applyfunc(sp.simplify)
dv.show("Dirac mass matrix at the aligned vacuum, m_D =", m_D)
assert (G3.T * m_D * G3 - m_D).applyfunc(sp.simplify) == sp.zeros(3, 3)
assert (G3.T * MR_inv * G3 - MR_inv).applyfunc(sp.simplify) == sp.zeros(3, 3)
ok("both m_D and M_R respect the residual Z2")

✓ all five Dirac structures on H-tilde pass feynlag's S3 check
S3-invariant M_R =   [1 terms]


⎡M₁₁   0    0 ⎤
⎢             ⎥
⎢ 0   M₁₁   0 ⎥
⎢             ⎥
⎣ 0    0   M₂₂⎦

✓ an S3-breaking Majorana entry — the check fires on a broken input, so it has teeth
✓ M_R = diag(M, M, M_S): two invariant structures, read back off the MajoranaBilinear Lagrangian


Dirac mass matrix at the aligned vacuum, m_D =   [1 terms]


⎡√2⋅Y_1__ν⋅v_S   √6⋅Y_2__ν⋅v₂         -√2⋅Y_2__ν⋅v₂          √6⋅Y_5__ν⋅v₂ ⎤
⎢───────────── + ────────────         ──────────────         ──────────── ⎥
⎢      2              6                     2                     6       ⎥
⎢                                                                         ⎥
⎢       -√2⋅Y_2__ν⋅v₂          √2⋅Y_1__ν⋅v_S   √6⋅Y_2__ν⋅v₂  √2⋅Y_5__ν⋅v₂ ⎥
⎢       ──────────────         ───────────── - ────────────  ──────────── ⎥
⎢             2                      2              6             2       ⎥
⎢                                                                         ⎥
⎢        √6⋅Y_4__ν⋅v₂                  √2⋅Y_4__ν⋅v₂          √2⋅Y_3__ν⋅v_S⎥
⎢        ────────────                  ────────────          ─────────────⎥
⎣             6                             2                      2      ⎦

✓ both m_D and M_R respect the residual Z2


---
## 6. Exact S₃ forbids solar and reactor mixing

If $G^{\mathsf T}m_DG=m_D$ and $G^{\mathsf T}M_RG=M_R$, then
$m_\nu=-m_DM_R^{-1}m_D^{\mathsf T}$ commutes with $G$ too. The charged-lepton and the light
neutrino rotations then both preserve the split into the odd direction and the even plane, and
so does $U_{\rm PMNS}=O_\ell^{\mathsf T}U_\nu$: the electron row has **one** non-zero entry.
Observed solar and reactor mixing need two of them. Checked symbolically on $m_\nu$, then
numerically on the PMNS matrix; the falsifier adds one $\mathbb Z_2$-odd Majorana entry.

In [ ]:
Msym, MSsym = sp.symbols("M M_S", positive=True)
MR_sym = MR_inv.subs({s: (Msym if str(s) == "M_00" else MSsym) for s in MR_inv.free_symbols})
m_nu_sym = (-m_D * MR_sym.inv() * m_D.T).applyfunc(sp.simplify)
assert (G3.T * m_nu_sym * G3 - m_nu_sym).applyfunc(sp.simplify) == sp.zeros(3, 3)
ok("the seesaw light mass matrix commutes with G for all Yukawas and M, M_S")

gnu = lfv.lepton_g_function(nus, m)
vevs0 = rl0["vevs"]
m_D_of = lambda Y: sum(v * gk for v, gk in zip(vevs0, gnu(Y)))
# R's columns at the symbolic vacuum, evaluated at this point's vacuum
subs0 = {m.v2.s: vevs0[1], m.vS.s: vevs0[2]}
w_odd_n = np.array(w_odd.subs(subs0).evalf(), dtype=float).ravel()
w_even = [np.array(R[:, k].subs(subs0).evalf(), dtype=float).ravel() for k in (0, 2)]


def M_R(M, MS, e1=0.0, e2=0.0):
    out = np.diag([M, M, MS]).astype(float)
    for e, w in ((e1, w_even[0]), (e2, w_even[1])):
        out += e * (np.outer(w_odd_n, w) + np.outer(w, w_odd_n))
    return out


rng = np.random.default_rng(9)
worst_row = 0.0
for _ in range(200):
    Y = rng.uniform(-1, 1, 5)
    mn, U = N.light_spectrum(m_D_of(Y), M_R(1e14, 1e14 * rng.uniform(0.3, 3)))
    row = np.sort(np.abs(O0.T @ U)[0])
    worst_row = max(worst_row, row[1])
print("largest second-largest |U_e i| over 200 random S3-invariant sectors: %.1e" % worst_row)
assert worst_row < 1e-12

Y = rng.uniform(-1, 1, 5)
row_broken = np.sort(np.abs(O0.T @ N.light_spectrum(m_D_of(Y), M_R(1e14, 2e14, 3e12))[1])[0])
print("with one Z2-odd Majorana entry: sorted |U_e i| =", np.round(row_broken, 5))
dv.falsify("the single-entry electron row with a Z2-odd M_R entry", lambda: row_broken[1] < 1e-12)

_ = ledger.step(r"Exact S₃ forces a single non-zero entry in the PMNS electron row: no solar or reactor mixing",
            obtained="derived",
            statement=r"G^{\mathsf T}m_DG=m_D,\ G^{\mathsf T}M_RG=M_R\ \Rightarrow\ [G,m_\nu]=0\ \Rightarrow\ |U_{ei}|=\delta_{i,\rm odd}",
            checks=("[G, m_nu] = 0 symbolically, for all five Yukawas and M, M_S",
                    "electron row single-entry to 1e-12 for 200 random S3-invariant sectors",
                    "the lepton analogue of V_us = 0 at exact S3 (notebook 02, finding 9)"),
            falsified_by="one Z2-odd Majorana entry gives two non-zero electron-row entries (falsify run)",
            section="5-6")

✓ the seesaw light mass matrix commutes with G for all Yukawas and M, M_S
largest second-largest |U_e i| over 200 random S3-invariant sectors: 8.8e-13
with one Z2-odd Majorana entry: sorted |U_e i| = [0.00201 0.00673 0.99998]
✓ the single-entry electron row with a Z2-odd M_R entry — the check fires on a broken input, so it has teeth


---
## 7. Where the breaking can live: the Majorana mass

The cheapest way out breaks S₃ only in $M_R$: a dimension-3, soft term that touches neither
the charged-lepton Yukawas nor the scalar potential, so the vacuum, the scalar spectrum and
every charged-lepton coupling of notebooks 03–08 stay exactly as they are. A general symmetric
$M_R$ has six entries; two are S₃-invariant. The other four split under the residual
$\mathbb Z_2$:

In [ ]:
Mgen = sp.Matrix(3, 3, lambda i, j: sp.Symbol(f"m_{min(i, j)}{max(i, j)}"))
M_even, M_odd = z2.symmetric_split(G3, Mgen)
syms = sorted(Mgen.free_symbols, key=str)
span = lambda Mx: sp.Matrix([list(Mx.diff(s_)) for s_ in syms]).rank()   # dimension of the image
rank_even, rank_odd = span(M_even), span(M_odd)
print("dimension of the Z2-even part: %d (of which 2 S3-invariant), Z2-odd part: %d" % (rank_even, rank_odd))
assert (rank_even, rank_odd) == (4, 2)
ok("of the four S3-breaking Majorana directions, two are Z2-even and two are Z2-odd")

dimension of the Z2-even part: 4 (of which 2 S3-invariant), Z2-odd part: 2
✓ of the four S3-breaking Majorana directions, two are Z2-even and two are Z2-odd


Only the two $\mathbb Z_2$-odd directions can populate the electron row. The fit below uses
**only** those, $\epsilon_1$ (odd–vacuum) and $\epsilon_2$ (odd–singlet), on top of the
invariant $M$, $M_S$.

**Inputs** (normal ordering): $\theta_{12}=33.68^\circ$, $\sin^2\theta_{13}=0.02215$,
$\theta_{23}=43.3^\circ$ (NuFIT 6.0, IC24 with SK-atm [NuFIT6]), $\Delta m^2_{21}=7.53\times10^{-5}$
eV² ([PDG]) and $\Delta m^2_{31}=2.5\times10^{-3}$ eV², the last an order-of-magnitude input:
its NuFIT 6.0 central value was not read off the table in this session, and the existence
result does not depend on it at the per-cent level. Real couplings, so no CP phase is fitted.

In [ ]:
TARGET = {"th12": 33.68, "s13": 0.02215, "th23": 43.3, "dm21": 7.53e-5, "dm31": 2.5e-3}
M1 = 1.0e-3          # lightest mass [eV]: a hierarchical benchmark (not fitted by oscillations)
EV = 1e-9
M_SCALE = 1.0e14


def observables(x, scale=M_SCALE):
    Y = x[:5] * math.sqrt(scale / M_SCALE)
    MR = M_R(scale, scale * math.exp(x[5]), scale * x[6], scale * x[7])
    mn, U = N.light_spectrum(m_D_of(Y), MR)
    th12, s13, th23 = N.pmns_angles(O0.T @ U)
    return mn / EV, th12, s13, th23


def residuals(x, penalty=0.0):
    mn, th12, s13, th23 = observables(x)
    r = [(mn[1]**2 - mn[0]**2) / TARGET["dm21"] - 1, (mn[2]**2 - mn[0]**2) / TARGET["dm31"] - 1,
         th12 - TARGET["th12"], (s13 - TARGET["s13"]) / 1e-3, th23 - TARGET["th23"], (mn[0] - M1) / 1e-4]
    return r + ([penalty * x[6], penalty * x[7]] if penalty else [])


rng = np.random.default_rng(2026)
fits = []
for k in range(80):
    x0 = np.r_[rng.uniform(-1, 1, 5), rng.uniform(-1, 1), rng.uniform(-0.5, 0.5, 2)]
    r = least_squares(residuals, x0, xtol=1e-15, ftol=1e-15, gtol=1e-15, max_nfev=5000)
    if r.cost < 1e-20:
        fits.append(r.x)
print("%d of 80 random starts reach an exact fit (m1 pinned at %.0e eV)" % (len(fits), M1))
assert fits
eps = np.array([math.hypot(x[6], x[7]) for x in fits])
generic = fits[int(np.argmin(eps))]           # a raw exact fit: no pressure on the breaking
best = generic
mn, th12, s13, th23 = observables(best)
print("smallest Z2-odd breaking among the raw exact fits: |eps|/M = %.3f" % eps.min())
print("  light masses [eV]: %s   theta12 = %.2f deg, sin2 theta13 = %.5f, theta23 = %.2f deg"
      % (np.round(mn, 5), th12, s13, th23))

# how small can the odd breaking go? penalize it and watch the fit residual
shrink = []
x_pen = best
for lam_pen in (1e-3, 1e-2, 1e-1):
    r = least_squares(lambda x: residuals(x, lam_pen), x_pen, xtol=1e-15, ftol=1e-15, gtol=1e-15, max_nfev=20000)
    x_pen = r.x
    fit_cost = 0.5 * sum(v * v for v in residuals(r.x))
    shrink.append((lam_pen, math.hypot(r.x[6], r.x[7]), fit_cost, observables(r.x)[0]))
    print("  penalty %.0e: |eps|/M = %.3f, data residual %.1e, light masses [eV] %s"
          % (*shrink[-1][:3], np.round(shrink[-1][3], 4)))
eps_min = min(e for _, e, c, _m in shrink if c < 1e-6)
print("smallest odd breaking found that still fits the data (residual < 1e-6): |eps|/M = %.1e" % eps_min)
best = x_pen                     # the benchmark: hierarchical, smallest breaking found
mn, th12, s13, th23 = observables(best)
print("benchmark: light masses [eV] %s, theta12 = %.2f, sin2 theta13 = %.5f, theta23 = %.2f, |eps|/M = %.1e"
      % (np.round(mn, 5), th12, s13, th23, math.hypot(best[6], best[7])))
ok("a Z2-odd soft breaking of M_R alone reproduces the oscillation data exactly")

_ = ledger.step("NuFIT 6.0 / PDG oscillation inputs (normal ordering)", obtained="imported",
            checks=(r"$\theta_{12}$, $\sin^2\theta_{13}$, $\theta_{23}$ and $\Delta m^2_{21}$ checked against web-search excerpts of [NuFIT6] and [PDG]",
                    r"$\Delta m^2_{31}=2.5\times10^{-3}$ eV² is an order-of-magnitude input, not a quoted best fit"),
            falsified_by="the published Table 1 differing from these inputs beyond their uncertainties",
            section="7")
_ = ledger.step(r"A $\mathbb Z_2$-odd soft breaking of $M_R$ alone reproduces the oscillation data",
            obtained="derived",
            checks=("exact fit of 5 oscillation observables with m1 = 1 meV, from 80 random starts at M = 1e14 GeV",
                    "a penalized search keeps the fit exact down to |eps|/M = 5e-4",
                    "only the two Z2-odd Majorana directions are switched on; Yukawas, potential and vacuum untouched",
                    "with them off the electron row is single-entry (section 6), so they are necessary"),
            falsified_by="no exact fit with only the Z2-odd Majorana directions",
            section="7-8")

69 of 80 random starts reach an exact fit (m1 pinned at 1e-03 eV)
smallest Z2-odd breaking among the raw exact fits: |eps|/M = 0.657
  light masses [eV]: [0.001   0.00873 0.05001]   theta12 = 33.68 deg, sin2 theta13 = 0.02215, theta23 = 43.30 deg


  penalty 1e-03: |eps|/M = 0.041, data residual 6.2e-14, light masses [eV] [0.001  0.0087 0.05  ]


  penalty 1e-02: |eps|/M = 0.004, data residual 4.1e-14, light masses [eV] [0.001  0.0087 0.05  ]


  penalty 1e-01: |eps|/M = 0.001, data residual 3.1e-13, light masses [eV] [0.001  0.0087 0.05  ]
smallest odd breaking found that still fits the data (residual < 1e-6): |eps|/M = 5.2e-04
benchmark: light masses [eV] [0.001   0.00873 0.05001], theta12 = 33.68, sin2 theta13 = 0.02215, theta23 = 43.30, |eps|/M = 5.2e-04
✓ a Z2-odd soft breaking of M_R alone reproduces the oscillation data exactly


---
## 8. Rescaling: one fit for every seesaw scale

$m_\nu=-m_DM_R^{-1}m_D^{\mathsf T}$ is invariant under $m_D\to s\,m_D$, $M_R\to s^2M_R$, so the
fit holds for any $M$ with $Y\propto\sqrt{M}$: the oscillation data fix $m_D^2/M$, not $M$.
This is what makes §9 a statement about every seesaw scale at once.

In [ ]:
for scale in (1e3, 1e8, 1e14):
    mn_s, th12_s, s13_s, th23_s = observables(best, scale)
    assert np.allclose(mn_s, mn, rtol=1e-8) and abs(th12_s - th12) < 1e-8
ok("the fit is scale-free under Y -> sqrt(M) Y, M_R -> M_R: checked at M = 1e3, 1e8, 1e14 GeV")

✓ the fit is scale-free under Y -> sqrt(M) Y, M_R -> M_R: checked at M = 1e3, 1e8, 1e14 GeV


---
## 9. Does the Majorana breaking leak into charged-lepton LFV?

At tree level, no: $M_R$ enters neither the charged-lepton couplings nor the scalar sector.
At one loop, μ→eγ picks up two new pieces through the heavy neutrinos $N$:

* **$H^\pm$–$N$ loops.** The Dirac Yukawas couple $\bar e_L N_R H^-$; with a neutral fermion of
  mass $m_N$ in the loop the amplitude is $A=-\sum_N Y_{eN}Y^*_{\mu N}f(t_N)/(32\pi^2m_{H^\pm}^2)$,
  $t_N=m_N^2/m_{H^\pm}^2$. The function $f$ is checked here three ways: against its
  Feynman-parameter integral, against the validated massless result of notebook 07
  ($f(0)=1/6$ reproduces `radiative.one_loop_charged` exactly), and for decoupling
  ($f\to1/(3t)$).
* **$W$–$N$ loops,** through the light–heavy mixing $V\simeq m_DM_R^{-1}$. Bounded here
  rather than computed: with a loop function of order one,
  $\mathcal B\lesssim\frac{3\alpha}{32\pi}\big(\sum_N|U_{eN}U_{\mu N}|\big)^2$.

In [ ]:
for t in (1e-8, 0.3, 1.0, 3.0, 1e4):
    assert abs(N.charged_scalar_loop(t) / N.charged_scalar_loop_integral(t) - 1) < 1e-8
rng = np.random.default_rng(0)
Yc_test = rng.normal(size=(3, 3))
AL_code, _ = RD.one_loop_charged(Yc_test, 400.0)
AL_mine = -(Yc_test.T @ Yc_test)[0, 1] / (32 * math.pi**2 * 400.0**2) * N.charged_scalar_loop(0.0)
assert abs(AL_code.real / AL_mine - 1) < 1e-12
assert abs(N.charged_scalar_loop(1e12) * 3e12 - 1) < 1e-6
ok("f(t): closed form = integral; f(0) reproduces notebook 07's massless charged loop; f -> 1/(3t)")
_ = ledger.step(r"Charged-scalar dipole function for a neutral fermion of any mass",
            obtained="derived",
            statement=r"f(t)=\int_0^1\frac{x^2(1-x)}{x+(1-x)t}dx=\frac{1-6t+3t^2+2t^3-6t^2\ln t}{6(1-t)^4}",
            checks=("closed form = Feynman-parameter integral to 1e-8", "f(0) reproduces radiative.one_loop_charged exactly",
                    "f -> 1/(3t): a heavy neutrino decouples"),
            falsified_by="a massless limit differing from notebook 07's validated charged loop",
            section="9")


def lfv_from_neutrinos(x, scale):
    Y = x[:5] * math.sqrt(scale / M_SCALE)
    MR = M_R(scale, scale * math.exp(x[5]), scale * x[6], scale * x[7])
    wM, UN = np.linalg.eigh(MR)
    mN = np.abs(wM)
    gk = gnu(Y)
    B_H = 0.0
    for a, key in ((1, "Hpm1"), (2, "Hpm2")):
        Yc = sum(rl0["R_C"][k, a] * math.sqrt(2) * gk[k] for k in range(3))
        Ym = O0.T @ Yc @ UN
        mH = rl0["mass"][key]
        A = -sum(Ym[0, n] * Ym[1, n] * N.charged_scalar_loop((mN[n] / mH) ** 2) for n in range(3)) / (32 * math.pi**2 * mH**2)
        B_H += RD.branching_ratio(A, 0.0, ALPHA, G_F)   # summed as a bound (no interference)
    Vmix = O0.T @ m_D_of(Y) @ np.linalg.inv(MR) @ UN
    B_W = 3 * ALPHA / (32 * math.pi) * sum(abs(Vmix[0, n] * Vmix[1, n]) for n in range(3)) ** 2
    return B_H, B_W, float(np.abs(Vmix).max())


MEG = 1.5e-13
leaks = {}
for label, x in (("generic exact fit", generic), ("smallest-breaking benchmark", best)):
    print("\n%s (|eps|/M = %.1e):" % (label, math.hypot(x[6], x[7])))
    print("%10s %12s %12s %12s %12s" % ("M [GeV]", "B(H+-N)", "B(W-N) bound", "max |V|", "sqrt(m3/M)"))
    leaks[label] = {}
    for scale in (1e3, 1e6, 1e10, 1e14):
        leaks[label][scale] = lfv_from_neutrinos(x, scale)
        print("%10.0e %12.1e %12.1e %12.1e %12.1e" % (scale, *leaks[label][scale], math.sqrt(observables(x)[0][2] * EV / scale)))
leak = leaks["smallest-breaking benchmark"]
worst_generic = max(max(v[0], v[1]) for v in leaks["generic exact fit"].values())
worst_leak = max(max(v[0], v[1]) for L in leaks.values() for v in L.values())
print("\nlargest neutrino-induced B(mu -> e gamma): generic fit %.1e, any benchmark %.1e  (MEG II: %.1e)"
      % (worst_generic, worst_leak, MEG))
assert worst_leak < 1e-6 * MEG
ok("the Majorana-sector breaking leaks into mu -> e gamma at least six orders below MEG II, for M = 1e3 ... 1e14 GeV")

_ = ledger.step(r"The $M_R$ breaking leaks into $\mu\to e\gamma$ only through $m_\nu/M_R$-suppressed neutrino loops",
            obtained="derived",
            checks=("H+-N loops computed on the fit benchmark (point 0 of notebook 01, mu3 = 1 GeV) for M = 1e3 ... 1e14 GeV",
                    "W-N loops bounded with an O(1) loop function",
                    "generic fit: both below 1e-30; the smallest-breaking benchmark (mixing enhanced ~1e3 by a cancellation in m_nu): below 1e-19",
                    "MEG II limit 1.5e-13"),
            falsified_by="a seesaw scale at which either piece reaches the MEG II sensitivity",
            section="9")

✓ f(t): closed form = integral; f(0) reproduces notebook 07's massless charged loop; f -> 1/(3t)

generic exact fit (|eps|/M = 6.6e-01):
   M [GeV]      B(H+-N) B(W-N) bound      max |V|   sqrt(m3/M)
     1e+03      1.9e-31      1.1e-31      3.8e-07      2.2e-07
     1e+06      1.3e-35      1.1e-37      1.2e-08      7.1e-09
     1e+10      1.3e-43      1.1e-45      1.2e-10      7.1e-11
     1e+14      1.3e-51      1.1e-53      1.2e-12      7.1e-13

smallest-breaking benchmark (|eps|/M = 5.2e-04):
   M [GeV]      B(H+-N) B(W-N) bound      max |V|   sqrt(m3/M)
     1e+03      3.4e-43      6.3e-20      3.8e-04      2.2e-07
     1e+06      3.9e-37      6.3e-26      1.2e-05      7.1e-09
     1e+10      7.1e-32      6.3e-34      1.2e-07      7.1e-11
     1e+14      1.0e-39      6.3e-42      1.2e-09      7.1e-13

largest neutrino-induced B(mu -> e gamma): generic fit 1.9e-31, any benchmark 6.3e-20  (MEG II: 1.5e-13)
✓ the Majorana-sector breaking leaks into mu -> e gamma at least six orders b

**Recognise.** The odd breaking needed is small: with a hierarchical spectrum
($m_1=1$ meV) a penalized search keeps the fit exact down to $|\epsilon|/M\approx5\times10^{-4}$ (a value found, not a proven minimum; how natural such a point is has not been
assessed). The absolute mass scale is an input here, not a fit result.

The leak is small for a structural reason. The oscillation data fix $m_D^2/M$, so for a
generic point $|V|^2\sim m_\nu/M$ and the Dirac Yukawas obey $Y^2\sim m_\nu M/v^2$: every
heavy-neutrino loop pays $Y^2/M^2\sim m_\nu/(v^2M)$, and the generic fit stays below
$10^{-30}$. The smallest-breaking benchmark escapes that estimate: its mixing exceeds
$\sqrt{m_3/M}$ by about $10^3$, the signature of a cancellation inside $m_\nu$, and its
$W$–$N$ bound reaches $6\times10^{-20}$ at $M=1$ TeV. That is still six orders below MEG II. So
the residual-$\mathbb Z_2$ selection rule of §3 holds in the charged-lepton and scalar sectors
**up to $m_\nu/M_R$-suppressed effects** (or mildly enhanced ones at tuned points), while the
neutrino sector breaks it as much as oscillations require.

---
## 10. Deliverable

In [ ]:
out = Path("results/neutrino_fit.json")
payload = {
    "description": ("Type-I seesaw for the S3 3HDM (notebook 09): Dirac Yukawas on H-tilde (the five S3 structures) "
                    "and M_R = diag(M, M, M_S) + the two Z2-odd soft directions (eps1: odd-vacuum, eps2: odd-singlet), "
                    "at the exact-S3 vacuum of notebook 01's point 0 with the charged leptons on the draft branch at "
                    "mu3 = 1 GeV. The fit is scale-free: Y scales as sqrt(M)."),
    "inputs": {"theta12_deg": TARGET["th12"], "sin2_theta13": TARGET["s13"], "theta23_deg": TARGET["th23"],
               "dm21_eV2": TARGET["dm21"], "dm31_eV2": TARGET["dm31"],
               "sources": "NuFIT 6.0 IC24 w/ SK-atm NO (angles), PDG 2024 (dm21); dm31 order-of-magnitude"},
    "benchmark": {"M_GeV": M_SCALE, "Y_nu": [float(v) for v in best[:5]],
                  "M_S_over_M": float(math.exp(best[5])), "eps1_over_M": float(best[6]), "eps2_over_M": float(best[7]),
                  "light_masses_eV": [float(v) for v in mn], "theta12_deg": th12, "sin2_theta13": s13, "theta23_deg": th23},
    "lightest_mass_eV_input": M1,
    "n_exact_fits_of_80": len(fits),
    "min_odd_breaking_over_M": float(eps_min),
    "neutrino_induced_mu_e_gamma": {lab: {f"{s:.0e}": {"B_Hpm_N": v[0], "B_W_N_bound": v[1], "max_mixing": v[2]}
                                          for s, v in L.items()} for lab, L in leaks.items()},
}
out.write_text(json.dumps(payload, indent=1) + "\n")
print("wrote %s (%.1f kB)" % (out, out.stat().st_size / 1024))

wrote results/neutrino_fit.json (2.4 kB)


In [ ]:
ledger.table()
ledger.to_latex("derivations_09.tex", subtitle="the residual Z₂ selection rule, and the seesaw neutrino sector")

### Derivation ledger — 09 — the residual ℤ₂ and the seesaw neutrino sector

**1. The residual $\mathbb Z_2$ of the aligned vacuum is the S₃ reflection $ab$ about the vacuum direction**  
<sub>derived here &nbsp;·&nbsp; §1</sub>

- ✓ found by testing all six S₃ elements on the aligned vacuum: exactly one non-identity element fixes it
- ✓ the potential is invariant under it (direct substitution)
- ✓ each soft pair splits into one $\mathbb Z_2$-even and one $\mathbb Z_2$-odd combination

*Would be falsified by:* a $\mathbb Z_2$-odd soft combination passing the invariance check (falsify run)


---

**2. Parities of the physical states: $e$, $h_0$, one $A$ and one $H^\pm$ are odd; $\mu$, $\tau$ and the other four scalars even**  
<sub>derived here &nbsp;·&nbsp; §2</sub>

- ✓ rotation columns tested as $G$-eigenvectors: (vacuum, in-plane, out-of-plane) = (+, −, +)
- ✓ $G^{\mathsf T}M_\ell G=M_\ell$ symbolically; the odd eigenvalue equals $m_e$ at a fitted point to $10^{-9}$

*Would be falsified by:* a mass eigenstate that is not a G-eigenvector (z2.parity raises)


---

**3. Selection rule: an amplitude whose external states multiply to $-1$ vanishes to all orders**  
<sub>derived here &nbsp;·&nbsp; §3-4</sub>

$$\prod_i P_i=-1\ \Rightarrow\ \mathcal A=0:\quad \mu\to e\gamma,\ \tau\to e\gamma,\ \mu\to3e,\ \mu N\to eN,\ h_{\rm even}\to e\ell,\ h_{\rm odd}\to\tau\mu \text{ forbidden}$$

- ✓ every odd-parity scalar-lepton coupling < 1e-14 over 594 exact (point, mu3) evaluations, parities read per eigenvector
- ✓ allowed off-diagonal couplings all non-zero (the test is not vacuous)
- ✓ notebook 08's B(tau -> e gamma) is round-off at all 118 380 exact evaluations

*Would be falsified by:* a non-zero coupling with odd total parity at any exact point


---

**4. Exact S₃ forces a single non-zero entry in the PMNS electron row: no solar or reactor mixing**  
<sub>derived here &nbsp;·&nbsp; §5-6</sub>

$$G^{\mathsf T}m_DG=m_D,\ G^{\mathsf T}M_RG=M_R\ \Rightarrow\ [G,m_\nu]=0\ \Rightarrow\ |U_{ei}|=\delta_{i,\rm odd}$$

- ✓ [G, m_nu] = 0 symbolically, for all five Yukawas and M, M_S
- ✓ electron row single-entry to 1e-12 for 200 random S3-invariant sectors
- ✓ the lepton analogue of V_us = 0 at exact S3 (notebook 02, finding 9)

*Would be falsified by:* one Z2-odd Majorana entry gives two non-zero electron-row entries (falsify run)


---

**5. NuFIT 6.0 / PDG oscillation inputs (normal ordering)**  
<sub>**imported** &nbsp;·&nbsp; §7</sub>

- ✓ $\theta_{12}$, $\sin^2\theta_{13}$, $\theta_{23}$ and $\Delta m^2_{21}$ checked against web-search excerpts of [NuFIT6] and [PDG]
- ✓ $\Delta m^2_{31}=2.5\times10^{-3}$ eV² is an order-of-magnitude input, not a quoted best fit

*Would be falsified by:* the published Table 1 differing from these inputs beyond their uncertainties


---

**6. A $\mathbb Z_2$-odd soft breaking of $M_R$ alone reproduces the oscillation data**  
<sub>derived here &nbsp;·&nbsp; §7-8</sub>

- ✓ exact fit of 5 oscillation observables with m1 = 1 meV, from 80 random starts at M = 1e14 GeV
- ✓ a penalized search keeps the fit exact down to |eps|/M = 5e-4
- ✓ only the two Z2-odd Majorana directions are switched on; Yukawas, potential and vacuum untouched
- ✓ with them off the electron row is single-entry (section 6), so they are necessary

*Would be falsified by:* no exact fit with only the Z2-odd Majorana directions


---

**7. Charged-scalar dipole function for a neutral fermion of any mass**  
<sub>derived here &nbsp;·&nbsp; §9</sub>

$$f(t)=\int_0^1\frac{x^2(1-x)}{x+(1-x)t}dx=\frac{1-6t+3t^2+2t^3-6t^2\ln t}{6(1-t)^4}$$

- ✓ closed form = Feynman-parameter integral to 1e-8
- ✓ f(0) reproduces radiative.one_loop_charged exactly
- ✓ f -> 1/(3t): a heavy neutrino decouples

*Would be falsified by:* a massless limit differing from notebook 07's validated charged loop


---

**8. The $M_R$ breaking leaks into $\mu\to e\gamma$ only through $m_\nu/M_R$-suppressed neutrino loops**  
<sub>derived here &nbsp;·&nbsp; §9</sub>

- ✓ H+-N loops computed on the fit benchmark (point 0 of notebook 01, mu3 = 1 GeV) for M = 1e3 ... 1e14 GeV
- ✓ W-N loops bounded with an O(1) loop function
- ✓ generic fit: both below 1e-30; the smallest-breaking benchmark (mixing enhanced ~1e3 by a cancellation in m_nu): below 1e-19
- ✓ MEG II limit 1.5e-13

*Would be falsified by:* a seesaw scale at which either piece reaches the MEG II sensitivity


---

<sub>8 steps: 7 derived here, 1 imported, 0 assumed</sub>

wrote derivations_09.tex  (8 steps)


'derivations_09.tex'

---
## 11. Findings

1. **The residual $\mathbb Z_2$ is the reflection $ab$** (§1). Of the six S₃ elements exactly one
   non-identity element fixes the aligned vacuum; the potential is invariant under it, and each
   soft pair splits into one $\mathbb Z_2$-even and one $\mathbb Z_2$-odd combination (the seed
   of the order parameter, roadmap B2).
2. **Parities, derived** (§2): the vacuum and out-of-plane directions are even and the in-plane
   non-vacuum direction is odd, in all three scalar sectors; $G^{\mathsf T}M_\ell G=M_\ell$, and
   the odd charged lepton is the electron (its mass equals $m_e$ at a fitted point to
   $10^{-9}$). So $e$, $h_0$, one $A$ and one $H^\pm$ are odd.
3. **The selection rule** (§3–§4): 8 of the 14 LFV channels listed are forbidden to all orders
   at the exact vacuum. Coupling by coupling, over 594 exact (point, $\mu_3$) evaluations, every
   odd-parity coupling is below $3\times10^{-15}$ while every allowed off-diagonal coupling is
   at least $3\times10^{-6}$; notebook 08's $\mathcal B(\tau\to e\gamma)$ is $7\times10^{-34}$ at
   all 118 380 exact evaluations.
4. **Exact S₃ cannot produce solar or reactor mixing** (§5–§6). The three $\nu_R$ allow five
   Dirac structures and an invariant $M_R=\mathrm{diag}(M,M,M_S)$; both respect the
   $\mathbb Z_2$, so $[G,m_\nu]=0$ and the PMNS electron row has a single non-zero entry
   (the second-largest is below $10^{-12}$ for 200 random sectors). This is the lepton analogue
   of $V_{us}=0$.
5. **A $\mathbb Z_2$-odd soft breaking of $M_R$ alone fixes it** (§7–§8). Of the four S₃-breaking
   Majorana directions two are $\mathbb Z_2$-odd; switching on only those reproduces the three
   angles and two splittings exactly with a hierarchical spectrum ($m_1=1$ meV), with
   $|\epsilon|/M$ down to $5\times10^{-4}$ found, and the fit holds at every seesaw scale ($Y\propto\sqrt M$).
6. **The breaking does not leak back** (§9). $H^\pm$–$N$ loops and a bound on $W$–$N$ loops give
   $\mathcal B(\mu\to e\gamma)$ below $10^{-30}$ for a generic fit, and at most $6\times10^{-20}$ at the
   smallest-breaking benchmark (whose mixing is enhanced by a cancellation in $m_\nu$), for $M$
   from $10^3$ to $10^{14}$ GeV: at least six orders below MEG II. The selection rule of the charged-lepton and scalar sectors therefore
   holds **up to $m_\nu/M_R$-suppressed effects**, and claim 1 of the roadmap stands with that
   qualifier.

**Caveats.** $\Delta m^2_{31}$ is an order-of-magnitude input; the PMNS fit is real (no
$\delta_{CP}$) and leaves the lightest mass is an input (1 meV); the $W$–$N$ piece is a bound with an
order-one loop function, not a computed rate. **Open:** the $\mathbb Z_2$-odd breaking could
instead sit in the scalar soft terms (vacuum misalignment), which would tie $\theta_{13}$ to
μ→eγ; that link is not explored here.

In [ ]:
check_quoted("11", [
    ("forbidden channels", len(forbidden), 8, 0),
    ("channels listed", len(table), 14, 0),
    ("exact (point, mu3) evaluations", n_eval, 594, 0),
    ("largest odd-parity coupling below 3e-15", worst_forbidden < 3e-15, True, 0),
    ("smallest allowed coupling at least 3e-6", smallest_allowed >= 3e-6, True, 0),
    ("nb08 max B(tau -> e gamma) / 1e-34", ex["max_br_tau_e"] / 1e-34, 7, 0.5),
    ("nb08 exact evaluations", ex["n_evaluations"], 118380, 0),
    ("second electron-row entry below 1e-12", worst_row < 1e-12, True, 0),
    ("some random starts fit exactly", len(fits) >= 10, True, 0),
    ("smallest odd breaking found / 1e-4", eps_min / 1e-4, 5.2, 0.5),
    ("generic-fit leak below 1e-30", worst_generic < 1e-30, True, 0),
    ("largest leak / 1e-20", worst_leak / 1e-20, 6.3, 0.5)])

✓ the 12 numbers quoted in §11 match this run


---
## 12. References

- **[LFVHD]** M. Zeleny-Mora, M. Mondragón, T. A. Valencia-Pérez, *"Exploring LFV Higgs decays in the
  Three Higgs Doublet Model"*, draft (`paper_lfvhd/LFVHD_3HDMS3.tex`): the S₃ assignment of the
  leptons and of the three $\nu_R$.
- **[NuFIT6]** I. Esteban, M. C. González-García, M. Maltoni, I. Martínez-Soler, J. P. Pinheiro,
  T. Schwetz, *"NuFit-6.0: updated global analysis of three-flavor neutrino oscillations"*,
  JHEP **12** (2024) 216, [arXiv:2410.05380](https://arxiv.org/abs/2410.05380): the normal-ordering
  angles (IC24 with SK-atm).
- **[PDG]** Particle Data Group, *Review of Particle Physics*, neutrino mixing listing
  ([pdg.lbl.gov](https://pdg.lbl.gov)): $\Delta m^2_{21}$.
- **[Lavoura03]** L. Lavoura, *"General formulae for $f_1\to f_2\gamma$"*, Eur. Phys. J. C **29**, 191
  (2003), [hep-ph/0302221](https://arxiv.org/abs/hep-ph/0302221): the class of loop functions §9's
  $f(t)$ belongs to; here it is checked against its own integral and notebook 07's massless limit.